<img src="images/banner.png" style="width: 100%;">

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1" # Enable in M1 Mac CPUs

from matplotlib import rcParams
import matplotlib.pyplot as plt


# Some preambles for prettification
rcParams.update({'figure.figsize': (8, 6), 'axes.spines.top': False,
                 'axes.spines.right': False, 'axes.labelsize': 12,
                 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                 'lines.linewidth': 1.5})

Reference: Chollet, F., & Watson, M. (2026). Deep learning with Python Third Edition. Manning.

Prepared by: Leodegario Lorenzo II

# Tokenization

In this notebook, we'll demonstrate several ways of tokenizing text.

In [2]:
# !pip install regex

In [3]:
import regex as re

CHAR_REGEX = re.compile(r'.')
WORD_REGEX = re.compile(r'[\w]+|[.,!?;]')

## 1 Character Tokenizer

### Tokenizer Definition

In [4]:
class CharTokenizer:
    """Tokenizer class that performs character-level tokenization"""

    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    
    def standardize(self, inputs):
        """Perform text standardization by case normalization"""
        return inputs.lower()


    def split(self, inputs):
        """Split input text by character"""
        return CHAR_REGEX.findall(inputs)


    def index(self, tokens):
        """Return the index of the given tokens"""
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]


    def __call__(self, inputs):
        """Perform the char tokenizer pipeline for the given input"""
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)

        return indices

### Obtaining the Character-level Vocabulary

In [5]:
import collections

In [6]:
def compute_char_vocabulary(inputs, max_size):
    """Return the vocabulary and counts for the given input corpus"""
    char_counts = collections.Counter()

    for x in inputs:
        x = x.lower()
        tokens = CHAR_REGEX.findall(x)
        char_counts.update(tokens)

    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)

    return dict((token, i) for i, token in enumerate(vocabulary))

### Sample Data

In [7]:
with open('data/moby.txt', 'r') as f:
    moby_dick = f.readlines()

In [8]:
char_vocabulary = compute_char_vocabulary(moby_dick, max_size=100)

In [9]:
print("Vocabulary length:", len(char_vocabulary))
print("Vocabulary start:", list(char_vocabulary.keys())[:10])
print("Vocabulary end:", list(char_vocabulary.keys())[-10:])

Vocabulary length: 69
Vocabulary start: ['[UNK]', ' ', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h']
Vocabulary end: ['#', 'ח', 'ן', 'ϰ', 'η', 'τ', 'ο', 'ς', 'œ', '%']


In [10]:
char_tokenizer = CharTokenizer(char_vocabulary)

In [11]:
print("Line length:", len(char_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 63


## 2 Word-level Tokenizer

### Tokenizer Definition

In [12]:
class WordTokenizer:
    """Tokenizer class that performs character-level tokenization"""

    def __init__(self, vocabulary):
        """Tokenizer class that performss word-level tokenization"""
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]


    def standardize(self, inputs):
        """Perform text standardization by case normalization"""
        return inputs.lower()


    def split(self, inputs):
        """Split input text by word"""
        return WORD_REGEX.findall(inputs)


    def index(self, tokens):
        """Return the index of the given tokens"""
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]


    def __call__(self, inputs):
        """Perform the word tokenizer pipeline for the given input"""
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)

        return indices

### Obtaining the Word-level Vocabulary

In [13]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()

    for x in inputs:
        x = x.lower()
        tokens = WORD_REGEX.findall(x)
        word_counts.update(tokens)

    vocabulary = ["[UNK]"]
    most_common = word_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)

    return dict((token, i) for i, token in enumerate(vocabulary))

### Sample Data

In [14]:
word_vocabulary = compute_word_vocabulary(moby_dick, max_size=2_000)

In [15]:
print("Vocabulary length:", len(word_vocabulary))
print("Vocabulary start:", list(word_vocabulary.keys())[:10])
print("Vocabulary end:", list(word_vocabulary.keys())[-10:])

Vocabulary length: 2000
Vocabulary start: ['[UNK]', ',', 'the', '.', 'of', 'and', 'a', 'to', 'in', ';']
Vocabulary end: ['fleet', 'aught', 'alike', 'worn', 'features', 'barbaric', 'helmsman', 'temporary', 'indispensable', 'fashion']


In [16]:
word_tokenizer = WordTokenizer(word_vocabulary)

In [17]:
print("Line length:", len(word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 13


## 3 Subword Tokenization

In [18]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

### Helper Functions

#### Character and Word Splitter

In [19]:
def count_and_split_words(data):
    counts = collections.Counter()

    for line in data:
        line = line.lower()
        for word in re.findall(r"[\w]+|[.,!?;]", line):
            chars = re.findall(r".", word)
            split_word = " ".join(chars)
            counts[split_word] += 1

    return dict(counts)

In [20]:
counts = count_and_split_words(data)
counts

{'t h e': 3,
 'q u i c k': 2,
 'b r o w n': 3,
 'f o x': 2,
 's l o w': 1,
 'f o x h o u n d': 1}

#### Pair Counter

In [21]:
def count_pairs(counts):
    pairs = collections.Counter()

    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq

    return pairs

In [22]:
pairs = count_pairs(counts)
pairs

Counter({('o', 'w'): 4,
         ('t', 'h'): 3,
         ('h', 'e'): 3,
         ('b', 'r'): 3,
         ('r', 'o'): 3,
         ('w', 'n'): 3,
         ('f', 'o'): 3,
         ('o', 'x'): 3,
         ('q', 'u'): 2,
         ('u', 'i'): 2,
         ('i', 'c'): 2,
         ('c', 'k'): 2,
         ('s', 'l'): 1,
         ('l', 'o'): 1,
         ('x', 'h'): 1,
         ('h', 'o'): 1,
         ('o', 'u'): 1,
         ('u', 'n'): 1,
         ('n', 'd'): 1})

In [23]:
first, second = max(pairs, key=pairs.get)

In [24]:
first, second

('o', 'w')

#### Pair Merger

In [25]:
def merge_pair(counts, first, second):
    split = re.compile(f"(?<!\S){first} {second}(?!\S)")
    merged = f"{first}{second}"

    return {split.sub(merged, word): count for word, count in counts.items()}

In [26]:
merge_pair(counts, first, second)

{'t h e': 3,
 'q u i c k': 2,
 'b r ow n': 3,
 'f o x': 2,
 's l ow': 1,
 'f o x h o u n d': 1}

### Obtaining the Subword Vocabulary

In [27]:
def compute_sub_word_vocabulary(dataset, vocab_size):
    counts = count_and_split_words(dataset)

    char_counts = collections.Counter()
    for word in counts:
        for char in word.split():
            char_counts[char] += counts[word]
    most_common = char_counts.most_common()
    vocab = ["[UNK]"] + [char for char, freq in most_common]
    merges = []

    while len(vocab) < vocab_size:
        pairs = count_pairs(counts)
        if not pairs:
            break
        first, second = max(pairs, key=pairs.get)
        counts = merge_pair(counts, first, second)
        vocab.append(f"{first}{second}")
        merges.append(f"{first} {second}")

    vocab = dict((token, index) for index, token in enumerate(vocab))
    merges = dict((token, rank) for rank, token in enumerate(merges))

    return vocab, merges

In [28]:
subword_vocab, merges = compute_sub_word_vocabulary(data, 2_000)

In [29]:
subword_vocab

{'[UNK]': 0,
 'o': 1,
 'h': 2,
 'w': 3,
 'n': 4,
 't': 5,
 'e': 6,
 'u': 7,
 'b': 8,
 'r': 9,
 'f': 10,
 'x': 11,
 'q': 12,
 'i': 13,
 'c': 14,
 'k': 15,
 's': 16,
 'l': 17,
 'd': 18,
 'ow': 19,
 'th': 20,
 'the': 21,
 'br': 22,
 'brow': 23,
 'brown': 24,
 'fo': 25,
 'fox': 26,
 'qu': 27,
 'qui': 28,
 'quic': 29,
 'quick': 30,
 'sl': 31,
 'slow': 32,
 'foxh': 33,
 'foxho': 34,
 'foxhou': 35,
 'foxhoun': 36,
 'foxhound': 37}

In [30]:
merges

{'o w': 0,
 't h': 1,
 'th e': 2,
 'b r': 3,
 'br ow': 4,
 'brow n': 5,
 'f o': 6,
 'fo x': 7,
 'q u': 8,
 'qu i': 9,
 'qui c': 10,
 'quic k': 11,
 's l': 12,
 'sl ow': 13,
 'fox h': 14,
 'foxh o': 15,
 'foxho u': 16,
 'foxhou n': 17,
 'foxhoun d': 18}

### Subword Tokenizer

In [31]:
class SubWordTokenizer:
    """Tokenizer class that performs subword-level tokenization"""

    def __init__(self, vocabulary, merges):
        self.vocabulary = vocabulary
        self.merges = merges
        self.unk_id = vocabulary["[UNK]"]


    def standardize(self, inputs):
        return inputs.lower()


    def bpe_merge(self, word):
        """Performs the byte-pair encoding based on word and merge
        vocabulary for a given word (with space bet. characters)
        """
        while True:
            # Get all the pairs in the current word
            pairs = re.findall(r"(?<!\S)\S+ \S+(?!\S)", word, overlapped=True)
            if not pairs:
                break

            # Check if there is still a pair that needs to be merged
            best = min(pairs, key=lambda pair: self.merges.get(pair, 1e9))
            if best not in self.merges:
                break

            # If there is, merge the pair
            first, second = best.split()
            split = re.compile(f"(?<!\S){first} {second}(?!\S)")
            merged = f"{first}{second}"

            # Update the current word splitting
            word = split.sub(merged, word)

        return word


    def split(self, inputs):
        """Splits the input text wrt to the merge vocabulary"""
        tokens = []

        for word in re.findall(r"[\w]+|[.,!?;]", inputs):
            word = " ".join(re.findall(r".", word))
            word = self.bpe_merge(word)
            tokens.extend(word.split())

        return tokens


    def index(self, tokens):
        """Get the intger index of the given tokens"""
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]


    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)

        return indices

### Sample Data

In [32]:
subword_vocabulary, subword_merges = compute_sub_word_vocabulary(
    moby_dick, 2_000
)

In [33]:
print("Vocabulary length:", len(subword_vocabulary))
print("Vocabulary start:", list(subword_vocabulary.keys())[:10])
print("Vocabulary end:", list(subword_vocabulary.keys())[-10:])

Vocabulary length: 2000
Vocabulary start: ['[UNK]', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h', 'r']
Vocabulary end: ['certainly', 'sor', 'similar', 'aught', 'fic', 'cul', 'aside', 'fles', 'gain', 'mood']


In [34]:
sub_word_tokenizer = SubWordTokenizer(subword_vocabulary, subword_merges)

In [35]:
print("Line length:", len(sub_word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

Line length: 16


<img src="images/banner-down.png" style="width: 100%;">